# PROVE BEFORE TRADE — Strict Feature Selection + Paper Trading
**เงินจริงยังไม่ถูกใช้** Notebook นี้คัดตัวแปรด้วยสถิติแบบเข้มก่อน แล้วจึงใช้ผลเป็นหลักฐานสำหรับขั้นต่อไป


In [ ]:
!pip -q install requests==2.32.5 pandas==2.2.3 "numpy>=1.26,<3" "yfinance>=0.2.54,<1" "statsmodels>=0.14.4,<1"


In [ ]:
import os
REPO='https://github.com/gujibata2-dotcom/adaptive-market-intelligence.git'
BRANCH='feat/paper-trading-bot'
if not os.path.exists('/content/adaptive-market-intelligence'):
    !git clone -b $BRANCH $REPO /content/adaptive-market-intelligence
%cd /content/adaptive-market-intelligence/paper_bot
!python test_core.py
!python test_proof_gate.py
!python test_feature_selection.py


## 1) Feature Lab V2 — ตัดตัวแปรที่ไม่ผ่านนัยสำคัญ


In [ ]:
from bot.feature_lab import run_feature_lab
scores = run_feature_lab()
display(scores)


In [ ]:
import json, pandas as pd
with open('/content/adaptive-market-intelligence/experiments/selected_features.json', encoding='utf-8') as f:
    selected = json.load(f)
print('SELECTED FEATURES:')
print(json.dumps(selected, ensure_ascii=False, indent=2))
rejected = pd.read_csv('/content/adaptive-market-intelligence/experiments/rejected_features.csv')
print('\nREJECTED FEATURES:')
display(rejected[['feature','target','train_p_fdr','test_p','same_direction','reason']])


## 2) ตรวจราคาจริง BTC/THB จาก Bitkub


In [ ]:
from bot.market import get_last_price
price = get_last_price('BTC_THB')
print('BTC/THB:', price)


## 3) Live Gate — ยังคงล็อกเงินจริง


In [ ]:
from bot.proof_gate import evaluate_live_gate
current_metrics = {
    'paper_days': 0,
    'closed_trades': 0,
    'net_return_pct': 0,
    'max_drawdown_pct': 0,
    'profit_factor': 0,
    'out_of_sample_pass': False,
    'walk_forward_pass': False,
    'fee_slippage_stress_pass': False,
    'regimes_tested': 0,
    'regime_robustness_pass': False,
    'prediction_ledger_locked': False,
}
evaluate_live_gate(current_metrics)


## 4) Paper Trading — เงินจำลอง 1,000 บาท


In [ ]:
from bot.engine import TradingEngine
engine = TradingEngine(symbol='BTC_THB', start_thb=1000, fast=5, slow=20, log_path='/content/trading_log.csv')
print('PAPER ENGINE READY — NO REAL ORDERS')
